# 06 Skills and ClawHub Lifecycle (OpenClaw, 2026)

## What This Lesson Is
Treat skills as versioned AI capabilities with precedence, validation, and rollout controls.

## Scientific Lens
- Concept: Skill governance relies on predictable precedence and validation before activation.
- Measure: Skill resolution correctness across workspace/local/bundled scopes.
- Validity Limit: Precedence rules alone cannot guarantee semantic skill quality.


## How It Works
1. Model precedence resolution rules deterministically.
2. Validate conflict/shadowing scenarios.
3. Run live OpenClaw prompt to generate a skill rollout checklist.


In [ ]:
import os
from openai import OpenAI  # OpenAI SDK used as protocol client to OpenClaw gateway

OPENCLAW_BASE_URL = os.getenv("OPENCLAW_BASE_URL", "http://127.0.0.1:18789").rstrip("/")
OPENCLAW_GATEWAY_TOKEN = os.getenv("OPENCLAW_GATEWAY_TOKEN") or os.getenv("OPENAI_API_KEY") or ""
OPENCLAW_TOKEN_SOURCE = (
    "OPENCLAW_GATEWAY_TOKEN" if os.getenv("OPENCLAW_GATEWAY_TOKEN")
    else ("OPENAI_API_KEY" if os.getenv("OPENAI_API_KEY") else "<missing>")
)
OPENCLAW_AGENT_ID = os.getenv("OPENCLAW_AGENT_ID", "main")

print("OPENCLAW_BASE_URL:", OPENCLAW_BASE_URL)
print("OPENCLAW_GATEWAY_TOKEN source:", OPENCLAW_TOKEN_SOURCE)
print("OPENCLAW_AGENT_ID:", OPENCLAW_AGENT_ID)


def build_gateway_client() -> OpenAI:
    # OpenClaw exposes an OpenAI-compatible Chat Completions endpoint at /v1/chat/completions.
    # We use the OpenAI SDK as a transport/protocol client to OpenClaw (not directly to OpenAI).
    # OpenClaw then routes to configured downstream providers/models.
    # Docs: https://docs.openclaw.ai/gateway/openai-http-api
    return OpenAI(base_url=f"{OPENCLAW_BASE_URL}/v1", api_key=OPENCLAW_GATEWAY_TOKEN or "local-dev-token")


def ask_openclaw(prompt: str, user: str = "lesson-user", temperature: float = 0.2) -> str:
    client = build_gateway_client()
    resp = client.chat.completions.create(
        model="openclaw",  # gateway-level alias/router target
        messages=[{"role": "user", "content": prompt}],
        user=user,
        temperature=temperature,
        extra_headers={"x-openclaw-agent-id": OPENCLAW_AGENT_ID},
    )
    return resp.choices[0].message.content or ""


### Why This Uses `OpenAI` Client With `model="openclaw"`
- The `OpenAI` SDK here is used as a **protocol-compatible HTTP client**.
- Requests go to the **OpenClaw gateway** (`OPENCLAW_BASE_URL/v1`), not directly to OpenAI.
- `model="openclaw"` is a **gateway alias/router target**.
- OpenClaw establishes downstream provider connections (OpenAI/Ollama/etc.) based on its own model/policy config.


## Code Walkthrough
- `Deterministic Demo` defines and validates the decision logic.
- `Live Demo` executes a real OpenClaw agent call through the OpenAI-compatible gateway API.


In [ ]:
# Deterministic Demo
skills = [
    {"name":"repo-review","scope":"bundled"},
    {"name":"repo-review","scope":"workspace"},
    {"name":"incident-brief","scope":"local"},
]
order = {"workspace":3, "local":2, "bundled":1}
resolved = {}
for s in skills:
    if s["name"] not in resolved or order[s["scope"]] > order[resolved[s["name"]]["scope"]]:
        resolved[s["name"]] = s
assert resolved["repo-review"]["scope"] == "workspace"


In [ ]:
# Live Demo
try:
    q = "Provide a 5-step skill rollout checklist covering validation, shadowing checks, and rollback."
    print(ask_openclaw(q, user="skills-governance"))
except Exception as exc:
    print(f"Live demo call failed: {exc}")
    print("Set OPENCLAW_GATEWAY_TOKEN in .env (or export OPENAI_API_KEY) and rerun.")


## Applied Labs
1. Add semantic version comparison in precedence decisions.
2. Create a canary rollout model with failure thresholds.
3. Define required observability fields for each skill release.

## Validation Checklist
- Precedence resolution is deterministic and test-backed.
- Skill rollout includes explicit governance controls.
- Live call focuses on operational skill lifecycle planning.

## Further Reading
- OpenClaw skills docs: https://docs.openclaw.ai/skills
- OpenClaw showcase docs: https://docs.openclaw.ai/start/showcase
